# ESG Sentiment Analyser — FinBERT (Distilled) + SHAP

**Transparent, explainable sentiment analysis on ESG-related financial text.**

---

## Research Motivation

Berg et al. (2022) showed that ESG ratings from major agencies diverge significantly due to methodological opacity. Rather than relying on these opaque third-party scores, this pipeline extracts ESG sentiment **directly from source documents**.

**Model choice:** `mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis` — a lightweight model fine-tuned specifically on financial news sentiment. Chosen over general-purpose BERT because domain-specific language matters in finance — words like 'exposure', 'risk', 'position' carry different meaning in financial vs general text.

**Key design principle:** Explainability by design — SHAP shows WHY each prediction was made, not just what it was.

## Step 1 — Install Dependencies

In [1]:
!pip install transformers torch shap pandas matplotlib -q
print('All libraries installed!')

All libraries installed!


## Step 2 — Import Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import shap
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import warnings
import os

warnings.filterwarnings('ignore')
os.makedirs('outputs', exist_ok=True)

print('Libraries loaded successfully!')
print(f'PyTorch version: {torch.__version__}')

Libraries loaded successfully!
PyTorch version: 2.2.2


## Step 3 — ESG Headlines Dataset

In a real research pipeline these would be scraped from sustainability reports, news APIs, or regulatory filings. For this prototype we use curated sample headlines across all three ESG dimensions.

In [3]:
ESG_HEADLINES = [
    # Environmental
    "Company reduces carbon emissions by 40% ahead of 2030 target",
    "Oil giant faces record fine for illegal dumping in protected wetlands",
    "Renewable energy firm secures 2 billion green bond for solar expansion",
    "Factory pollution levels breach environmental safety thresholds",
    "Tech company commits to 100 percent renewable energy across all data centres",
    # Social
    "Retailer improves gender pay gap reporting with full transparency",
    "Mining company faces lawsuit over unsafe working conditions",
    "Bank launches financial inclusion programme for underserved communities",
    "Pharma firm accused of price gouging on life-saving medication",
    "Firm wins award for industry-leading parental leave policy",
    # Governance
    "CEO ousted after board discovers undisclosed conflicts of interest",
    "Company publishes full supply chain audit for first time",
    "Regulator investigates firm for misleading ESG disclosure",
    "Board adopts independent oversight committee for ethical AI use",
    "Audit reveals significant gaps in corporate governance framework",
]

ESG_CATEGORIES = [
    "Environmental", "Environmental", "Environmental", "Environmental", "Environmental",
    "Social", "Social", "Social", "Social", "Social",
    "Governance", "Governance", "Governance", "Governance", "Governance",
]

print(f'Dataset: {len(ESG_HEADLINES)} ESG headlines loaded')
print(f'Categories: E={ESG_CATEGORIES.count("Environmental")} S={ESG_CATEGORIES.count("Social")} G={ESG_CATEGORIES.count("Governance")}')

Dataset: 15 ESG headlines loaded
Categories: E=5 S=5 G=5


## Step 4 — Load Financial Sentiment Model

Using `mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis` — a distilled model fine-tuned on financial news. Lightweight (~80MB vs ~500MB for full FinBERT) but trained on domain-specific financial text, making it far more appropriate than general-purpose BERT for ESG signal extraction.

In [4]:
MODEL_NAME = "mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis"

print(f'Loading financial sentiment model...')
print(f'Model: {MODEL_NAME}')
print('(Downloading ~80MB on first run)')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()

# Get label mapping from model config
id2label = model.config.id2label
print(f'\nModel loaded! Labels: {list(id2label.values())}')

COLORS = {'positive': '#4a7c59', 'negative': '#c85c2a', 'neutral': '#a07000'}
EMOJIS = {'positive': 'positive', 'negative': 'negative', 'neutral': 'neutral'}

Loading financial sentiment model...
Model: mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis
(Downloading ~80MB on first run)


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/933 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/328M [00:00<?, ?B/s]


Model loaded! Labels: ['negative', 'neutral', 'positive']


## Step 5 — Predict Sentiment

In [1]:
def predict_sentiment(texts):
    results = []
    for text in texts:
        inputs = tokenizer(
            text, return_tensors='pt',
            truncation=True, max_length=128, padding=True
        )
        with torch.no_grad():
            outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1).squeeze()
        predicted_id = torch.argmax(probs).item()
        label = id2label[predicted_id].lower()
        results.append({
            'text': text,
            'sentiment': label,
            'confidence': round(probs[predicted_id].item() * 100, 1),
            'scores': {id2label[i].lower(): round(p.item() * 100, 1)
                      for i, p in enumerate(probs)}
        })
    return results

print('Running sentiment analysis...')
results = predict_sentiment(ESG_HEADLINES)
print('Done!\n')

for i, r in enumerate(results):
    cat = ESG_CATEGORIES[i]
    print(f'[{cat}] {r["sentiment"].upper()} ({r["confidence"]}%)')
    print(f'  "{r["text"]}"')
    print(f'  Scores: {r["scores"]}')
    print()

Running sentiment analysis...


NameError: name 'ESG_HEADLINES' is not defined

## Step 6 — Build Results DataFrame

In [ ]:
rows = []
for i, r in enumerate(results):
    row = {
        'text': r['text'],
        'esg_category': ESG_CATEGORIES[i],
        'sentiment': r['sentiment'],
        'confidence': r['confidence'],
    }
    row.update(r['scores'])
    rows.append(row)

df = pd.DataFrame(rows)
print(df[['esg_category', 'sentiment', 'confidence']].to_string(index=False))

## Step 7 — Visualise Sentiment Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('ESG Sentiment Analysis — Domain-Specific Financial NLP + SHAP',
             fontsize=13, fontweight='bold')

# Overall
sc = df['sentiment'].value_counts()
bc = [COLORS.get(s, '#888') for s in sc.index]
axes[0].bar(sc.index, sc.values, color=bc, edgecolor='white', width=0.5)
axes[0].set_title('Overall Sentiment Distribution')
axes[0].set_ylabel('Headlines')
for i, v in enumerate(sc.values):
    axes[0].text(i, v + 0.05, str(v), ha='center', fontweight='bold')

# By category
cs = df.groupby(['esg_category', 'sentiment']).size().unstack(fill_value=0)
cc = [COLORS.get(c, '#888') for c in cs.columns]
cs.plot(kind='bar', ax=axes[1], color=cc, edgecolor='white')
axes[1].set_title('Sentiment by ESG Category')
axes[1].set_ylabel('Headlines')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Sentiment')

plt.tight_layout()
plt.savefig('outputs/sentiment_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/sentiment_distribution.png')

## Step 8 — Confidence per Headline

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
bc = [COLORS.get(s, '#888') for s in df['sentiment']]
lbls = [t[:60] + '...' if len(t) > 60 else t for t in df['text']]

ax.barh(range(len(df)), df['confidence'], color=bc, edgecolor='white')
ax.set_yticks(range(len(df)))
ax.set_yticklabels(lbls, fontsize=8)
ax.set_xlabel('Confidence (%)')
ax.set_title('Model Confidence per ESG Headline', fontweight='bold')
ax.axvline(x=70, color='gray', linestyle='--', alpha=0.5)
ax.set_xlim(0, 110)

legend_elements = [
    mpatches.Patch(facecolor='#4a7c59', label='Positive'),
    mpatches.Patch(facecolor='#c85c2a', label='Negative'),
    mpatches.Patch(facecolor='#a07000', label='Neutral')
]
ax.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()
plt.savefig('outputs/confidence_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/confidence_scores.png')

## Step 9 — SHAP Explainability

This is the **core white-box contribution** — making the model's reasoning visible.

SHAP assigns a contribution score to each word:
- **Positive value** → word pushed prediction toward current label
- **Negative value** → word pushed prediction away from current label

This makes every prediction **auditable and challengeable** — directly aligned with EU AI Act requirements.

In [ ]:
def predict_fn(texts):
    all_probs = []
    for text in texts:
        inputs = tokenizer(
            text, return_tensors='pt',
            truncation=True, max_length=128, padding=True
        )
        with torch.no_grad():
            outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1).squeeze().numpy()
        all_probs.append(probs)
    return np.array(all_probs)

output_names = [id2label[i].lower() for i in range(len(id2label))]
masker = shap.maskers.Text(tokenizer)
explainer = shap.Explainer(predict_fn, masker, output_names=output_names)

print('SHAP explainer ready!')
print(f'Output labels: {output_names}')

In [ ]:
# Explain one headline at a time to keep memory low
# Environmental — positive
headline_1 = [ESG_HEADLINES[0]]
print(f'Explaining: "{headline_1[0]}"')
print(f'Predicted: {results[0]["sentiment"].upper()} ({results[0]["confidence"]}%)')
print('Generating SHAP values (takes ~30 seconds)...')

shap_1 = explainer(headline_1)
shap.plots.text(shap_1[0])
print('Done!')

In [ ]:
# Governance — negative
headline_2 = [ESG_HEADLINES[10]]
print(f'Explaining: "{headline_2[0]}"')
print(f'Predicted: {results[10]["sentiment"].upper()} ({results[10]["confidence"]}%)')
print('Generating SHAP values...')

shap_2 = explainer(headline_2)
shap.plots.text(shap_2[0])
print('Done!')

In [ ]:
# Social — positive
headline_3 = [ESG_HEADLINES[5]]
print(f'Explaining: "{headline_3[0]}"')
print(f'Predicted: {results[5]["sentiment"].upper()} ({results[5]["confidence"]}%)')
print('Generating SHAP values...')

shap_3 = explainer(headline_3)
shap.plots.text(shap_3[0])
print('Done!')

## Step 10 — Save Results

In [ ]:
df.to_csv('outputs/esg_sentiment_results.csv', index=False)
print('Results saved: outputs/esg_sentiment_results.csv')
print()
print('=== PIPELINE COMPLETE ===')
print(f'Headlines analysed: {len(df)}')
print(f'Sentiment breakdown: {df["sentiment"].value_counts().to_dict()}')
print()
print('Output files:')
print('  outputs/sentiment_distribution.png')
print('  outputs/confidence_scores.png')
print('  outputs/esg_sentiment_results.csv')